In [2]:
# Install dependencies
!pip install yfinance pandas numpy plotly --quiet

# Imports
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# Factor ETFs and Labels
factor_etfs = {
    "VLUE": "Value",
    "MTUM": "Momentum",
    "SPLV": "Low Volatility",
    "SPY": "Benchmark"
}

# Top Stocks by Factor
factor_top_stocks = {
    "Value": ["JPM", "BRK-B", "META"],
    "Momentum": ["NVDA", "JPM", "AMZN"],
    "Low Volatility": ["MSFT", "BRK-B", "JPM"]
}

# Date Range
start_date = "2015-01-01"
end_date = datetime.today().strftime('%Y-%m-%d')

# Download Data
print("📥 Downloading ETF data...")
raw_etf_data = yf.download(list(factor_etfs.keys()), start=start_date, end=end_date, auto_adjust=True)
etf_data = raw_etf_data['Close'] if 'Close' in raw_etf_data.columns else raw_etf_data

print("📥 Downloading stock data...")
all_stocks = list({s for stocks in factor_top_stocks.values() for s in stocks})
raw_stock_data = yf.download(all_stocks, start=start_date, end=end_date, auto_adjust=True)
stock_data = raw_stock_data['Close'] if 'Close' in raw_stock_data.columns else raw_stock_data

# Resample to Monthly
etf_monthly = etf_data.resample("M").last()
stock_monthly = stock_data.resample("M").last()
etf_returns = etf_monthly.pct_change()

# Strategy Logic
strategy_returns = []
best_factors_history = []

for i in range(1, len(etf_returns)):
    prev_month = etf_returns.iloc[i - 1].drop("SPY", errors='ignore')
    if prev_month.isnull().all():
        continue

    best_factor_etf = prev_month.idxmax()
    best_factor_name = factor_etfs.get(best_factor_etf)
    if best_factor_name is None:
        continue

    best_factors_history.append(best_factor_name)
    stocks_to_buy = factor_top_stocks.get(best_factor_name, [])

    try:
        returns = (stock_monthly[stocks_to_buy].iloc[i] / stock_monthly[stocks_to_buy].iloc[i - 1]) - 1
        avg_return = returns.mean()
        strategy_returns.append(avg_return)
    except:
        continue

# Performance Series
strategy_returns = pd.Series(strategy_returns, index=etf_returns.index[1:len(strategy_returns)+1])
benchmark_returns = etf_returns["SPY"].loc[strategy_returns.index]

cumulative_strategy = (1 + strategy_returns).cumprod()
cumulative_benchmark = (1 + benchmark_returns).cumprod()

# Performance Metrics
def sharpe_ratio(returns, risk_free=0.01):
    excess = returns - (risk_free / 12)
    return np.mean(excess) / np.std(excess) * np.sqrt(12)

def max_drawdown(series):
    peak = series.cummax()
    drawdown = (series / peak) - 1
    return drawdown.min()

strategy_total_return = round((cumulative_strategy[-1] - 1) * 100, 1)
benchmark_total_return = round((cumulative_benchmark[-1] - 1) * 100, 1)
cagr = (cumulative_strategy[-1]) ** (1 / (len(strategy_returns)/12)) - 1
volatility = strategy_returns.std() * np.sqrt(12)
sharpe = sharpe_ratio(strategy_returns)
drawdown = max_drawdown(cumulative_strategy) * 100
latest_month = strategy_returns.index[-1].strftime('%B %Y')
latest_factor = best_factors_history[-1] if best_factors_history else "N/A"
latest_picks = ", ".join(factor_top_stocks.get(latest_factor, []))

# Strategy Summary (HTML-Formatted)
summary_text = (
    f"<b>Total Return:</b> {strategy_total_return:.1f}% (Strategy) vs {benchmark_total_return:.1f}% (SPY Benchmark)<br>"
    f"<b>Annualized Sharpe Ratio:</b> {sharpe:.3f}<br>"
    f"<b>Maximum Drawdown:</b> {drawdown:.2f}%<br>"
    f"<b>CAGR:</b> {cagr*100:.2f}%<br>"
    f"<b>Annualized Volatility:</b> {volatility*100:.2f}%<br><br>"

    f"<b>📅 Stock Picks for {latest_month}:</b><br>"
    f"<b>Best Performing Factor:</b> {latest_factor}<br>"
    f"<b>Buy:</b> {latest_picks}<br><br>"

    f"<b>Momentum Top 3:</b> {', '.join(factor_top_stocks['Momentum'])}<br>"
    f"<b>Volatility Top 3:</b> {', '.join(factor_top_stocks['Low Volatility'])}<br>"
    f"<b>Value Top 3:</b> {', '.join(factor_top_stocks['Value'])}<br><br>"

    "<b>📌 Strategy Logic:</b> Monthly factor rotation based on trailing 1-month ETF performance. "
    "The top factor ETF guides stock selection from preselected proxies.<br><br>"
    "<b>📚 Theoretical Justification:</b> Empirical studies (e.g., Asness et al. 2013) validate factor persistence. "
    "Our model aligns with this by adaptively capturing the dominant signal. Returns are enhanced via concentrated equity exposure rather than broad ETF allocation.<br><br>"
    "<b>⚠️ Assumptions:</b> No transaction costs, taxes, or liquidity frictions. Stocks selected manually per factor. Results reflect backtest only.<br><br>"
    "<b>🧠 Conclusion:</b> This project demonstrates how quantitative finance leverages simple momentum and cross-sectional alpha concepts to achieve meaningful market outperformance."
)

# Build Interface
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.7, 0.3],
    specs=[[{}], [{"type": "table"}]],
    vertical_spacing=0.05
)

# Add Strategy Line
fig.add_trace(go.Scatter(
    x=cumulative_strategy.index,
    y=cumulative_strategy,
    name="Strategy",
    line=dict(color="#00008B", width=3)
), row=1, col=1)

# Add SPY Benchmark Line
fig.add_trace(go.Scatter(
    x=cumulative_benchmark.index,
    y=cumulative_benchmark,
    name="SPY Benchmark",
    line=dict(color="#40826D", width=3)  # Solid muted green
), row=1, col=1)

# Summary Text as Table
fig.add_trace(go.Table(
    columnwidth=[3],
    header=dict(values=["<b>Strategy Summary</b>"], fill_color='black', font=dict(size=16, color='white'), align='left'),
    cells=dict(values=[[summary_text]], fill_color='black', align='left', font=dict(size=12, color='white'))
), row=2, col=1)

# Layout
fig.update_layout(
    title="📈 Adaptive Factor Rotation Strategy vs SPY (Backtest)",
    xaxis_title="Date",
    yaxis_title="Cumulative Return (%)",
    yaxis_tickformat=".1%",
    height=1000,
    template="plotly_dark",
    margin=dict(l=40, r=40, t=80, b=40),
    showlegend=True
)

fig.show()


📥 Downloading ETF data...


[*********************100%***********************]  4 of 4 completed


📥 Downloading stock data...


[*********************100%***********************]  6 of 6 completed
/tmp/ipython-input-2-2621728425.py:42: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

/tmp/ipython-input-2-2621728425.py:43: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.

/tmp/ipython-input-2-2621728425.py:87: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-2-2621728425.py:88: FutureWarning:

Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`

/tmp/ipython-input-2-2621728425.py:89: FutureWarning:

Series.__getitem__ treating keys as positions is d